# code

> fill kosha from a repo, grep the working tree, and federate all of it with the prose

In [ ]:
#| default_exp code

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

[kosha](https://github.com/vedicreader/kosha) is the code half of the vault: AST chunks, symbol
names, and a call graph with PageRank over your repo and your installed packages. It embeds code
with a code-trained model, deliberately — identifiers are not sentences — so the two stores share
no vector space, and `federate` fuses their *rankings* rather than their distances.
[rgapi](https://github.com/AnswerDotAI/rgapi) adds a third kind of evidence for free: ripgrep over
the files as they are on disk right now, which is the only leg that sees what nothing has indexed.

In [ ]:
#| export
from fastcore.all import AttrDict, L, Path, patch
from litesearch import rrf_all
from vishalakshi.core import Vault

In [ ]:
#| export
@patch
def kosha(self:Vault,
          dir:str=None,              # repo root; defaults to the cwd repo
          share_encoder:bool=False,  # embed code with the vault's encoder instead of kosha's
          **kw                       # forwarded to Kosha()
):
    """The `Kosha` for `dir`, cached on the vault so `index_code` and `code_search` hit one store.

    `share_encoder=True` makes the two stores directly comparable at the cost of worse code
    ranking; it needs a real encoder, so it is a no-op on the hashing fallback."""
    from kosha import Kosha
    k, root = getattr(self, '_k', None), Path(dir).resolve() if dir else None
    if k is None or (root and k.root != root):
        if share_encoder and self.enc.model: kw['efn'] = lambda: self.enc.model
        self._k = k = Kosha(dir, **kw)
    return k

@patch
def index_code(self:Vault,
               dir:str=None,      # repo to index
               graph:bool=True,   # also build the AST call graph (callers, callees, PageRank)
               env:bool=False,    # also index installed packages (slow the first time)
               force:bool=False,
               verbose:bool=False,
               **kw               # forwarded to kosha update_repo/sync
) -> dict:
    """Point the vault at a repo and fill kosha's stores from it.

    This is the code path proper — symbol search and call-graph navigation — as opposed to
    `Vault.code()`, which files source files into the vault as ordinary documents. Both can
    coexist, and `federate()` searches whichever exist."""
    k = self.kosha(dir)
    if env: k.sync(dir=dir, force=force, verbose=verbose, sync_graph=graph, in_parallel=True, **kw)
    else:
        k.update_repo(dir, force=force, verbose=verbose, **kw)
        if graph: k.graph.sync(dir=str(dir or k.root), force=force)
    return k.status()

In [ ]:
#| export
@patch
def code_search(self:Vault, q:str, limit:int=10, dir:str=None, **kw) -> L:
    """Search code through kosha: FTS + ANN over repo and environment, fused and rank-boosted.
    Supports kosha's `key:value` filters, so `'retry package:httpx'` and `'lang:py chunker'` work."""
    return self.kosha(dir).context(q, limit=limit, **kw)

@patch
def symbol(self:Vault, name:str, depth:int=1, dir:str=None) -> AttrDict:
    'A symbol in the call graph: its file, PageRank and degree, plus its callers and callees.'
    k = self.kosha(dir)
    return AttrDict(node=name, info=dict(k.ni(name) or {}), neighbors=L(k.neighbors(name, depth)))

@patch
def where_to_add(self:Vault, description:str, limit:int=5, dir:str=None) -> L:
    'Where in the indexed repo a described change belongs — kosha ranking over the call graph.'
    return self.kosha(dir).where_to_add(description, limit=limit)

@patch
def grep(self:Vault,
         pattern:str,       # a ripgrep regex
         dir:str='.',       # tree to search
         limit:int=20,      # matching lines returned
         **kw               # forwarded to rgapi.rg (glob=, ext=, context=, hidden=, ...)
) -> L:
    """Exact matches in the files on disk, through ripgrep.

    The leg neither of the others can serve: embeddings generalise and FTS5 stems, so an identifier
    that appears verbatim in a file the vault never ingested — or ingested an older copy of — is
    invisible to both. `.gitignore` applies, so build output stays out of the results."""
    from rgapi import rg
    return _rg(rg(pattern, root=dir, max_results=limit, smart_case=True, **kw))

In [ ]:
#| export
def _prose(v, q, n, kind=None) -> L:
    'Vault sections, normalised to the federated row shape.'
    return L(v.sections(q, limit=n, kind=kind)).map(
        lambda s: AttrDict(source='prose', ref=s['node_id'], title=s['title'], where=s['breadcrumb'],
                           text=' '.join(s['snippets'])[:600], open=f"read({s['node_id']!r})"))

def _code(rows, leg:str) -> L:
    'kosha hits, normalised to the federated row shape.'
    def row(r):
        m = dict(r.get('metadata') or {})
        mod, path, ln = m.get('mod_name', ''), m.get('path', ''), m.get('lineno')
        return AttrDict(source=leg, ref=mod or path, title=mod or (Path(path).name if path else ''),
                        where=f'{path}:{ln}' if path and ln else (path or mod),
                        text=(r.get('content') or '')[:600],
                        open=f'symbol({mod!r})' if mod else f'open {path}')
    return L(rows).map(row)

def _rg(matches) -> L:
    'ripgrep lines, normalised to the federated row shape.'
    return L(matches).map(lambda m: AttrDict(source='grep', ref=f'{m.path}:{m.line_number}',
                                             title=Path(m.path).name, where=f'{m.path}:{m.line_number}',
                                             text=m.line.strip()[:600], open=f'open {m.path}'))

@patch
def federate(self:Vault,
             q:str,             # the query
             limit:int=12,      # fused hits returned
             prose:bool=True,   # the vault's own documents, papers, notes
             repo:bool=True,    # kosha's repo index
             env:bool=False,    # kosha's installed-package index
             grep:bool=True,    # ripgrep over the working tree
             kind:str=None,     # restrict the prose leg to some KINDS
             weights:dict=None, # per-leg RRF weights, e.g. {'prose':1.0,'repo':1.5}
             dir:str=None,      # repo for the code and grep legs
             per_leg:int=None,  # hits pulled from each leg before fusion
) -> AttrDict:
    """One ranked answer across prose, indexed code and the files on disk, fused by RRF.

    The legs share no vector space — the vault embeds prose, kosha embeds identifiers, ripgrep
    embeds nothing — so they cannot be merged by distance. Reciprocal Rank Fusion needs only each
    leg's *ordering*, which is exactly what survives a change of encoder, and it is the same
    mechanism litesearch already uses to combine FTS with vectors. Each leg is tried
    independently: `legs` reports what each contributed, or why it did not."""
    n, legs = per_leg or max(limit, 10), {}
    def leg(nm, f):
        try: legs[nm] = f()
        except Exception as e: legs[f'{nm}_error'] = f'{type(e).__name__}: {str(e)[:120]}'
    if prose: leg('prose', lambda: _prose(self, q, n, kind=kind))
    if repo:  leg('repo', lambda: _code(self.kosha(dir).repo_context(q, limit=n), 'repo'))
    if env:   leg('env', lambda: _code(self.kosha(dir).env_context(q, limit=n), 'env'))
    if grep:  leg('grep', lambda: self.grep(q, dir or '.', limit=n))
    lists = {nm: rows for nm, rows in legs.items() if isinstance(rows, L) and rows}
    for nm, rows in lists.items():
        for i, r in enumerate(rows): r['_fid'] = f'{nm}:{r.ref or i}'
    fused = rrf_all(list(lists.values()), limit=limit, id_key='_fid',
                    weights=[(weights or {}).get(nm, 1.0) for nm in lists])
    return AttrDict(query=q, hits=L(fused).map(AttrDict),
                    legs={nm: (len(r) if isinstance(r, L) else r) for nm, r in legs.items()},
                    note=f"RRF over {', '.join(lists) or 'nothing'}; the legs use different "
                         f"encoders, so ranks are fused, not distances")

## Try it

Each leg is optional and each fails on its own: with no repo indexed, `legs` says so and the
others still answer.

In [ ]:
v = Vault(':memory:')
v.note('litesearch fuses the FTS and vector legs with reciprocal rank fusion.')
f = v.federate('rank fusion', repo=False, grep=False)
f.hits[0].where, f.legs, f.note

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


('litesearch fuses the FTS and vector legs with reciprocal rank fusion.',
 {'prose': 1},
 'RRF over prose; the legs use different encoders, so ranks are fused, not distances')

In [ ]:
test_eq(f.legs['prose'], 1)
test_eq(f.hits[0].source, 'prose')
test_eq(v.federate('rank fusion', prose=False, repo=False, grep=False).hits, [])  # every leg off: empty, not an error
assert v.grep('reciprocal rank fusion', '.')                                     # ripgrep reads the disk, not the vault